In [37]:
import sqlite3
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path.cwd()

CSV_PATH = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "corporate_emissions_clean.csv"
)

# SQLite database
DB_PATH = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "corporate_emissions.db")

print("Project directory:", PROJECT_DIR)
print("CSV exists:", CSV_PATH.is_file())
print("Database exists:", DB_PATH.is_file())

# Create a practice SQLite database
conn = sqlite3.connect(DB_PATH)


# Import the cleaned CSV
df = pd.read_csv(CSV_PATH)

df.to_sql(
    "emissions",
    conn,
    if_exists="replace",
    index=False
)

print("Data imported:", df.shape)

Project directory: /Users/molan/Library/CloudStorage/OneDrive-UniversityofHelsinki/career_transition/Python_learning/Corporate_Emissions_Project
CSV exists: True
Database exists: True
Data imported: (11, 8)


In [38]:
query = """
PRAGMA table_info(emissions);
"""

table_info = pd.read_sql(query, conn)

display(table_info)


,cid,name,type,notnull,dflt_value,pk
0,0,Activity Name,TEXT,0,None,0
1,1,Scope,INTEGER,0,None,0
2,2,Year,INTEGER,0,None,0
3,3,Business Unit,TEXT,0,None,0
4,4,Activity,INTEGER,0,None,0
5,5,Emission Factor,REAL,0,None,0
6,6,Emissions tCO2e,REAL,0,None,0
7,7,Report Date,TEXT,0,None,0


In [39]:
query = """
SELECT COUNT(*) AS number_of_records
FROM emissions;
"""

pd.read_sql(query, conn)

,number_of_records
0,11


In [40]:
query = """
SELECT
    Year,
    "Activity Name",
    Scope,
    "Business Unit",
    COUNT(*) AS record_count
FROM emissions

GROUP BY Year, "Activity Name", Scope, "Business Unit"

HAVING COUNT(*) > 1;
"""

pd.read_sql(query, conn)

,Year,Activity Name,Scope,Business Unit,record_count


In [41]:
query = """
-- Query 1: Calculate annual GHG emissions

SELECT
    Year,
    SUM("Emissions tCO2e") AS toal_emissions
FROM emissions
GROUP BY Year;

"""

pd.read_sql(query, conn)

,Year,toal_emissions
0,2024,481.00
1,2025,396.04


In [42]:
query = """

SELECT
    Year,scope,
    SUM("Emissions tCO2e") AS total_emissions_tco2e
FROM emissions
GROUP BY Year, scope
ORDER BY Year, scope;

"""

pd.read_sql(query, conn)

,Year,Scope,total_emissions_tco2e
0,2024,1,51.60
1,2024,2,25.00
2,2024,3,404.40
3,2025,1,19.44
4,2025,2,22.00
5,2025,3,354.60


In [43]:
query = """
WITH activity_summary AS (
    SELECT
        Year,
        "Activity Name",
        SUM("Emissions tCO2e") AS activity_emissions_tco2e
    FROM emissions
    GROUP BY Year, "Activity Name"
)

SELECT
    Year,
    "Activity Name",
    activity_emissions_tco2e,
    ROW_NUMBER() OVER (
        PARTITION BY Year 
        ORDER BY activity_emissions_tco2e DESC
        ) AS Rank

FROM activity_summary
ORDER BY Year, Rank;
"""
pd.read_sql(query, conn)

,Year,Activity Name,activity_emissions_tco2e,Rank
0,2024,Purchased steel,390.00,1
1,2024,Natural gas boiler,30.00,2
2,2024,Purchased electricity,25.00,3
3,2024,Company diesel trucks,21.60,4
4,2024,Business flights,8.40,5
5,2024,Employee commuting,6.00,6
6,2025,Purchased steel,342.00,1
7,2025,Purchased electricity,22.00,2
8,2025,Company diesel trucks,19.44,3
9,2025,Business flights,7.00,4


In [44]:

query = """
WITH annual_emissions AS (
    SELECT
        Year,
        SUM("Emissions tCO2e") AS annual_emissions_tco2e
    FROM emissions
    GROUP BY Year
)

SELECT
    Year,
    annual_emissions_tco2e,

    LAG(annual_emissions_tco2e) OVER (
        ORDER BY Year
    ) AS previous_year_emissions_tco2e

FROM annual_emissions
ORDER BY Year;
"""
pd.read_sql(query, conn)

,Year,annual_emissions_tco2e,previous_year_emissions_tco2e
0,2024,481.00,NaN
1,2025,396.04,481.0


In [45]:
query = """

WITH annual_emissions AS (
    SELECT
        Year,
        SUM("Emissions tCO2e") AS annual_emissions_tco2e
    FROM emissions
    GROUP BY Year
),

annual_with_previous AS (
    SELECT
        Year,
        annual_emissions_tco2e,
        LAG(annual_emissions_tco2e) OVER (
            ORDER BY Year
        ) AS previous_year_emissions_tco2e
    FROM annual_emissions
)

SELECT
    Year,
    annual_emissions_tco2e,
    previous_year_emissions_tco2e,
    100.0 * (annual_emissions_tco2e - previous_year_emissions_tco2e) / previous_year_emissions_tco2e 
    AS YoY_change
    
FROM annual_with_previous
ORDER BY Year;
"""
pd.read_sql(query, conn)

,Year,annual_emissions_tco2e,previous_year_emissions_tco2e,YoY_change
0,2024,481.00,NaN,NaN
1,2025,396.04,481.0,-17.663202


In [46]:
# Read SQL queries from the .sql file

SQL_PATH = PROJECT_DIR / "Corporate_Emissions_5_Queries.sql"

sql_text = SQL_PATH.read_text(encoding="utf-8")

# Split the file into individual queries
queries = [
    q.strip()
    for q in sql_text.split(";")
    if q.strip()
]

print("Number of queries:", len(queries))

Number of queries: 6


In [47]:
second_query = sql_text.split(";")[1]

result = pd.read_sql_query(second_query, conn)

display(result)

,Year,Scope,total_emissions_tco2e
0,2024,1,51.60
1,2024,2,25.00
2,2024,3,404.40
3,2025,1,19.44
4,2025,2,22.00
5,2025,3,354.60


In [48]:
# Test each SQL query independently

for i, sql in enumerate(queries, start=1):

    print(f"\n--- Query {i} ---")

    result = pd.read_sql_query(sql, conn)

    print("Rows returned:", len(result))

    display(result.head())


--- Query 1 ---
Rows returned: 2


,Year,total_emissions_tco2e
0,2024,481.00
1,2025,396.04



--- Query 2 ---
Rows returned: 6


,Year,Scope,total_emissions_tco2e
0,2024,1,51.60
1,2024,2,25.00
2,2024,3,404.40
3,2025,1,19.44
4,2025,2,22.00



--- Query 3 ---
Rows returned: 11


,Year,Activity Name,activity_emissions_tco2e,Rank
0,2024,Purchased steel,390.0,1
1,2024,Natural gas boiler,30.0,2
2,2024,Purchased electricity,25.0,3
3,2024,Company diesel trucks,21.6,4
4,2024,Business flights,8.4,5


... (2 outputs left out)


--- Query 5 ---
Rows returned: 2


,Year,annual_emissions_tco2e,previous_year_emissions_tco2e,YoY_change
0,2024,481.00,NaN,NaN
1,2025,396.04,481.0,-17.66



--- Query 6 ---
Rows returned: 0


,Year,Activity Name,Scope,Business Unit,record_count


In [49]:
print("SQL file:", SQL_PATH.resolve())

print("File exists:", SQL_PATH.exists())

print("File size:", SQL_PATH.stat().st_size)

print("First 200 characters:")
print(repr(SQL_PATH.read_text(encoding="utf-8")[:200]))

SQL file: /Users/molan/Library/CloudStorage/OneDrive-UniversityofHelsinki/career_transition/Python_learning/Corporate_Emissions_Project/Corporate_Emissions_5_Queries.sql
File exists: True
File size: 2706
First 200 characters:
'-- Query 1: Annual GHG Emissions\n-- Purpose: Calculate total GHG emissions by reporting year.\n-- Unit: tCO2e\nSELECT\n    Year,\n    SUM("Emissions tCO2e") AS total_emissions_tco2e\nFROM emissions\nGROUP B'


In [50]:
pd.read_sql_query("PRAGMA database_list;", conn)

,seq,name,file
0,0,main,/Users/molan/Library/CloudStorage/OneDrive-Uni...


In [51]:
pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type = 'table';",
    conn
)

,name
0,emissions


In [52]:
# Read the saved SQL file
sql_text = SQL_PATH.read_text(encoding="utf-8")

# Separate the five queries
queries = [
    q.strip()
    for q in sql_text.split(";")
    if q.strip()
]

print("Number of queries:", len(queries))

# Execute each query
for i, query in enumerate(queries, start=1):
    print(f"\n--- Query {i} ---")

    result = pd.read_sql_query(query, conn)

    print("Rows returned:", len(result))
    display(result)

Number of queries: 6

--- Query 1 ---
Rows returned: 2


,Year,total_emissions_tco2e
0,2024,481.00
1,2025,396.04



--- Query 2 ---
Rows returned: 6


,Year,Scope,total_emissions_tco2e
0,2024,1,51.60
1,2024,2,25.00
2,2024,3,404.40
3,2025,1,19.44
4,2025,2,22.00
5,2025,3,354.60



--- Query 3 ---
Rows returned: 11


,Year,Activity Name,activity_emissions_tco2e,Rank
0,2024,Purchased steel,390.00,1
1,2024,Natural gas boiler,30.00,2
2,2024,Purchased electricity,25.00,3
3,2024,Company diesel trucks,21.60,4
4,2024,Business flights,8.40,5
5,2024,Employee commuting,6.00,6
6,2025,Purchased steel,342.00,1
7,2025,Purchased electricity,22.00,2
8,2025,Company diesel trucks,19.44,3
9,2025,Business flights,7.00,4


... (2 outputs left out)


--- Query 5 ---
Rows returned: 2


,Year,annual_emissions_tco2e,previous_year_emissions_tco2e,YoY_change
0,2024,481.00,NaN,NaN
1,2025,396.04,481.0,-17.66



--- Query 6 ---
Rows returned: 0


,Year,Activity Name,Scope,Business Unit,record_count


In [53]:
scopes = ["Scope 1", "Scope 2", "Scope 3"]
for i, scope in enumerate(scopes, start=1):
    print(f"\n--- scope {i} for {scope} ---")



--- scope 1 for Scope 1 ---

--- scope 2 for Scope 2 ---

--- scope 3 for Scope 3 ---


In [55]:
from pathlib import Path
import json

project_dir = Path.cwd()

for path in project_dir.glob("*.ipynb"):
    notebook = json.loads(path.read_text(encoding="utf-8"))

    code = str([
        cell.get("source", [])
        for cell in notebook["cells"]
    ])

    outputs = str([
        cell.get("outputs", [])
        for cell in notebook["cells"]
    ])

    print("\nNotebook:", path.name)
    print("Absolute paths in code:", "/Users/" in code)
    print("Absolute paths in outputs:", "/Users/" in outputs)
    print("Saved output cells:", sum(
        bool(cell.get("outputs"))
        for cell in notebook["cells"]
    ))

print("\nRequirements file exists:",
      (project_dir / "requirements.txt").is_file())


Notebook: Cororate_Emissions_cleaning_pipline.ipynb
Absolute paths in code: False
Absolute paths in outputs: False
Saved output cells: 0

Notebook: Corporate_Emission_SQL_Analysis_v1.ipynb
Absolute paths in code: True
Absolute paths in outputs: False
Saved output cells: 0

Requirements file exists: True
